## Fast API Working Example

In FastAPI, the URL and the HTTP method are created at the same time, in a single line of code. The framework uses Python decorators to bind them together.

Below is a minimal, complete example and a step-by-step explanation of what actually happens.

```bash
from fastapi import FastAPI

# 1. Create the FastAPI app
# This creates an API server object.
# Think of it as an empty container that will hold all your endpoints.
app = FastAPI()
```

```bash
@app.get("/hello")
def say_hello():
    return {"Hello": "World"}
```


`@app.get("/hello")` does 3 things:

* Declares the HTTP method → GET
* Declares the URL path → /hello
* Registers this combination with FastAPI

At this point, FastAPI internally records something like:
```bash
GET /hello → say_hello()
```


### Attach the logic to the endpoint
```bash
def say_hello():
    return {"message": "Hello, world"}
```

This function is the handler:
* it runs when a request matches `GET /hello`
* its return value is automatically converted to JSON
* FastAPI also adds a 200 OK status code by default

### Start the Server (Terminal Command)
From the directory containing main.py, run:
```bash
uvicorn main:app --reload
```
* `uvicorn`: Starts the ASGI server
* `main`: Python file name (main.py)
* `app`: FastAPI object name (app = FastAPI())
* `--reload`: Auto-reloads the server when code changes (development only)

### Status Code:
* 200 OK: The request was successful. The server returns the requested data.
* 201 Created: The request was successful and a new resource was created (common for POST).
* 204 No Content: The request was successful, but there is no content to return.
* 400 Bad Request: The request was invalid (e.g., missing or malformed data).
* 401 Unauthorized: Authentication is required or failed.
* 403 Forbidden: You do not have permission to access this resource.
* 404 Not Found: The requested resource does not exist on the server.
* 422 Unprocessable Entity: The request was well-formed but contains invalid data (common with FastAPI validation errors).
* 500 Internal Server Error: The server encountered an error processing the request.

### Get Method

GET is idempotent (幂等性指的是：一次和多次请求某个资源产生的结果是一样的。)

In [1]:
import requests

# Server: http://127.0.0.1:8000
# Endpoint: hello
# Method: GET
response = requests.get("http://127.0.0.1:8000/hello")
print(response.json())

{'Hello': 'World'}


In [2]:
# Server: http://127.0.0.1:8000
# Endpoint: customers/123
# Method: GET
response = requests.get("http://127.0.0.1:8000/customers/123")
print(response.json())

{'customer_id': 123}


### Post Method

POST is not idempotent

In [3]:
data = {"name": "Book", "price": 12.5}
response = requests.post("http://127.0.0.1:8000/items", json=data)
print(response.status_code)
print(response.json())

200
{'message': 'Item created', 'item': {'name': 'Book', 'price': 12.5}}


In [4]:
data = {"name": "Book", "price": "alice"}
response = requests.post("http://127.0.0.1:8000/items", json=data)
print(response.status_code)
print(response.json())

422
{'detail': [{'type': 'float_parsing', 'loc': ['body', 'price'], 'msg': 'Input should be a valid number, unable to parse string as a number', 'input': 'alice'}]}


In [5]:
data = [1, 2, 3, 4]
response = requests.post("http://127.0.0.1:8000/sum", json=data)
print(response.status_code)
print(response.json())

200
{'sum': 10.0}


## FastAPI used as an Agent Endpoint (Example 1)

* FastAPI exposes an HTTP endpoint; inside that endpoint you call the OpenAI SDK.
* The “agent” lives inside the FastAPI handler.


The workflow layers:

> ```scss
> Client (browser / system)
>        ↓ HTTP
> FastAPI endpoint (agent interface)
>        ↓ Python
> OpenAI SDK (LLM reasoning)
>        ↓
> Response returned to client
> ```

* FastAPI is how the agent is invoked. 
* OpenAI SDK is how the agent thinks

### Step by Step:
**Step 1: FastAPI defines an agent interface through**
>> ```bash
>> @app.post("/agent", response_model=...)
>> ```

>> This creates `POST /agent`. This endpoint is how external systems call the agent

>> Adding response_model to validate the response data against the AgentResponse schema

**Step 2: Request becomes structured agent input**

```bash
class AgentRequest(BaseModel):
    question: str
```
FastAPI guarantees:

* JSON is valid
* question exists
* Type is correct

This is agent input validation

**Step 3: The agent reasons using OpenAI**
```bash
response = client.responses.create(
    model="gpt-4.1-mini",
    input=request.question
)
```
This is the agent’s brain:
* Receives task
* Produces reasoning + output
* Returns a structured response

**Step 4: Agent returns a clean response**

```bash
return AgentResponse(answer=answer)
```

FastAPI:
* Serializes to JSON
* Sends HTTP response
* Enforces schema

In [6]:
response = requests.post(
    "http://127.0.0.1:8000/agent",
    json={"question": "Explain what a credit risk model is"},
)
print(response.json().get("answer"))

Of course. Here’s a clear and comprehensive explanation of what a credit risk model is.

### Core Definition
A **credit risk model** is a mathematical framework or statistical tool used by financial institutions to **quantify the potential risk of loss**


## FastAPI used as an Agent Endpoint (Example 2)

Below is a minimal, concrete demonstration of tool-calling inside an agent, using:

* FastAPI → exposes the agent
* OpenAI Python SDK (Responses API) → agent reasoning
* A local Python function → the “tool”

The goal is to show how the LLM decides to call a tool, how your code executes it, and how the result flows back to the model.

Workflow Layers:
> ```scss
> Client
>   ↓ HTTP
> FastAPI endpoint (agent)
>   ↓
> LLM decides: "I need a tool"
>   ↓
> Your Python function runs
>   ↓
> Result sent back to LLM
>   ↓
> Final answer returned to client
> ```

**Remember: The LLM does not execute code. It asks you to run tools, and your agent runtime does the execution.**

In [7]:
response = requests.post(
    "http://127.0.0.1:8000/agent_tools",
    json={
        "question": "What is the monthly payment for a $500,000 loan at 6% over 30 years?"
    },
)

print(response.json())

{'answer': 'The monthly payment for a $500,000 loan at 6% interest over 30 years would be **$2,500.00**.\n\nThis calculation assumes a standard mortgage-style payment structure with equal monthly payments over the loan term.'}
